# Notebook 12: Revision Analyses

This notebook implements the full quantitative revision workflow for:
**"The External Exposome and Life Expectancy: Formaldehyde as a Leading Predictor in U.S. Counties"**.

It follows the structure and modeling logic of `11_xgboost_bayesian_optimization_run4.ipynb`, while adding the reviewer-driven analyses required for the major revision.

Key goals:
- fit the smoking-augmented revised model (Model B)
- evaluate whether formaldehyde remains notable after controlling for smoking
- test formaldehyde vs. socioeconomic confounding
- assess temporal generalization and spatial autocorrelation
- quantify SHAP ranking stability and Bayesian optimization convergence
- quantify CAMS coarse-resolution support
- assess livestock robustness via a no-livestock sensitivity model
- update the ablation study using Model B feature rankings

Explicitly out of scope for this notebook:
- OMI/TROPOMI direct validation
- FoT recomputation from raw atmospheric data


---
## Section 0: Dependency Check, Imports, and Notebook Helpers

This section checks that the required Python packages are available in the notebook kernel environment (expected: `main_env`), imports all libraries, defines plotting constants, establishes output paths, and provides small helper functions used throughout the notebook.


In [1]:
# Preflight dependency check for the active notebook kernel (expected: main_env)
import importlib.util

REQUIRED_PACKAGES = {
    'xgboost': 'xgboost',
    'shap': 'shap',
    'skopt': 'scikit-optimize',
    'geopandas': 'geopandas',
    'statsmodels': 'statsmodels',
    'libpysal': 'libpysal',
    'esda': 'esda'
}

missing_modules = [module_name for module_name in REQUIRED_PACKAGES
                   if importlib.util.find_spec(module_name) is None]

if missing_modules:
    missing_conda_packages = [REQUIRED_PACKAGES[module_name] for module_name in missing_modules]
    install_cmd = 'conda install -n main_env -c conda-forge ' + ' '.join(missing_conda_packages)
    print('=' * 70)
    print('MISSING NOTEBOOK DEPENDENCIES')
    print('=' * 70)
    print(f'Missing Python modules in the active kernel: {missing_modules}')
    print('\nInstall them in main_env with:')
    print(f'  {install_cmd}')
    raise ModuleNotFoundError(
        'Notebook dependencies are missing from the active kernel environment. '
        'Install the missing packages in main_env, then rerun this cell.'
    )
else:
    print('=' * 70)
    print('ALL NOTEBOOK DEPENDENCIES ARE AVAILABLE IN THE ACTIVE KERNEL')
    print('=' * 70)


ALL NOTEBOOK DEPENDENCIES ARE AVAILABLE IN THE ACTIVE KERNEL


In [2]:
# Core libraries and notebook configuration
import json
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import geopandas as gpd
import shap

from scipy.stats import pearsonr
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from skopt import BayesSearchCV
from skopt.space import Integer, Real
from libpysal.weights import Queen
from esda.moran import Moran

warnings.filterwarnings('ignore')

# Paths
BASE_DATA_PATH = Path('../data_cleaned/combined_final/final_combined_all_variables_reduced.csv')
SMOKING_DATA_PATH = Path('/Users/samyakshrestha/Projects/predicting-lung-cancer/data/processed/smoking_by_year/smoking_all_years.csv')
COUNTY_SHP_PATH = Path('../data/shapefiles/cb_2019_us_county_20m.shp')
OUTPUT_DIR = Path('../data_cleaned/outputs_cleaned/modeling/xgboost/revision')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Model constants
TARGET_COL = 'Mean Life Expectancy'
IDENTIFIER_COLS = ['County', 'State', 'Year', 'Fips']
FORMALDEHYDE_FEATURE = 'FoT Formaldehyde Above75ᵗʰ Percentile'
SMOKING_FEATURE = 'Smoking Rate'
REPORTING_EXCLUDE_FEATURES = ['is_post_2015']
LIVESTOCK_FEATURES = ['Cattle', 'Chicken', 'Duck', 'Goat', 'Horse', 'Pig', 'Sheep']
RUN4_BASELINE_TEST_R2 = 0.854
EXPECTED_MODEL_B_FEATURES = 45
BAYES_N_ITER = 50

# MDPI-style figure specifications (matching Run4)
SINGLE_COL_WIDTH = 3.27
DOUBLE_COL_WIDTH = 6.85
DPI = 600
FONT_SIZE = 11

plt.rcParams['font.size'] = FONT_SIZE
plt.rcParams['font.family'] = 'Arial'
sns.set_palette('husl')

# Bayesian search space (same as Run4)
SEARCH_SPACES = {
    'n_estimators': Integer(200, 1500),
    'max_depth': Integer(4, 8),
    'learning_rate': Real(0.01, 0.15, prior='log-uniform'),
    'subsample': Real(0.6, 0.95),
    'colsample_bytree': Real(0.5, 0.9),
    'reg_alpha': Real(0.01, 5.0, prior='log-uniform'),
    'reg_lambda': Real(0.1, 5.0, prior='log-uniform'),
    'min_child_weight': Integer(3, 15),
}

DISPLAY_REPLACEMENTS = {
    'Μm': 'µm',
}

artifact_manifest = []


def record_artifact(path, section, purpose, paper_facing):
    artifact_manifest.append({
        'path': str(Path(path)),
        'section': section,
        'purpose': purpose,
        'paper_facing': bool(paper_facing)
    })


def _json_safe(value):
    if isinstance(value, dict):
        return {k: _json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_safe(v) for v in value]
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    return value


def save_json(data, path):
    path = Path(path)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(_json_safe(data), f, indent=2)


def clean_display_labels(labels):
    def _clean_one(label):
        label = str(label)
        for old, new in DISPLAY_REPLACEMENTS.items():
            label = label.replace(old, new)
        return label

    if isinstance(labels, str):
        return _clean_one(labels)
    return [_clean_one(label) for label in labels]


def filter_reporting_features(feature_names):
    return [feature for feature in feature_names if feature not in REPORTING_EXCLUDE_FEATURES]


def prepare_xy(df, extra_drop=None):
    drop_cols = IDENTIFIER_COLS + [TARGET_COL]
    if extra_drop is not None:
        drop_cols = drop_cols + list(extra_drop)
    drop_cols = [col for col in drop_cols if col in df.columns]

    X = df.drop(columns=drop_cols)
    y = df[TARGET_COL]
    groups = df['Fips']
    return X, y, groups


def run_bayes_xgb(X_train, y_train, groups_train, section_name, n_iter=BAYES_N_ITER):
    print('=' * 70)
    print(f'RUNNING BAYESIAN OPTIMIZATION: {section_name}')
    print('=' * 70)
    print(f'Training rows: {len(X_train):,}')
    print(f'Number of features: {X_train.shape[1]}')
    print('Using 5-fold GroupKFold by county')

    model = xgb.XGBRegressor(
        objective='reg:squarederror',
        random_state=42,
        tree_method='hist'
    )

    bayes = BayesSearchCV(
        estimator=model,
        search_spaces=SEARCH_SPACES,
        n_iter=n_iter,
        cv=GroupKFold(n_splits=5),
        scoring='r2',
        n_jobs=-1,
        random_state=42,
        refit=True,
        verbose=1,
    )

    bayes.fit(X_train, y_train, groups=groups_train)

    print('\nOptimization complete.')
    print(f'Best CV R²: {bayes.best_score_:.4f}')
    print('Best parameters:')
    for key, value in bayes.best_params_.items():
        print(f'  - {key}: {value}')

    return bayes, bayes.best_estimator_


def build_fixed_param_model(best_params):
    return xgb.XGBRegressor(
        objective='reg:squarederror',
        random_state=42,
        tree_method='hist',
        **_json_safe(best_params)
    )


def evaluate_predictions(y_train, train_predictions, y_test, test_predictions, n_features):
    train_rmse = float(np.sqrt(mean_squared_error(y_train, train_predictions)))
    test_rmse = float(np.sqrt(mean_squared_error(y_test, test_predictions)))
    train_mae = float(mean_absolute_error(y_train, train_predictions))
    test_mae = float(mean_absolute_error(y_test, test_predictions))
    train_r2 = float(r2_score(y_train, train_predictions))
    test_r2 = float(r2_score(y_test, test_predictions))

    n_train = len(y_train)
    n_test = len(y_test)
    adj_r2_train = float(1 - (1 - train_r2) * ((n_train - 1) / (n_train - n_features - 1)))
    adj_r2_test = float(1 - (1 - test_r2) * ((n_test - 1) / (n_test - n_features - 1)))

    metrics = {
        'train_r2': train_r2,
        'test_r2': test_r2,
        'train_adj_r2': adj_r2_train,
        'test_adj_r2': adj_r2_test,
        'train_rmse': train_rmse,
        'test_rmse': test_rmse,
        'train_mae': train_mae,
        'test_mae': test_mae,
        'train_n': n_train,
        'test_n': n_test,
        'n_features': int(n_features),
    }
    return metrics


def save_metrics_table(metrics, path, title):
    metrics_df = pd.DataFrame({
        'Metric': ['R² Score', 'Adjusted R²', 'RMSE (years)', 'MAE (years)', 'Sample Size'],
        'Training Set': [
            f"{metrics['train_r2']:.3f}",
            f"{metrics['train_adj_r2']:.3f}",
            f"{metrics['train_rmse']:.2f}",
            f"{metrics['train_mae']:.2f}",
            f"{metrics['train_n']:,}",
        ],
        'Test Set': [
            f"{metrics['test_r2']:.3f}",
            f"{metrics['test_adj_r2']:.3f}",
            f"{metrics['test_rmse']:.2f}",
            f"{metrics['test_mae']:.2f}",
            f"{metrics['test_n']:,}",
        ]
    })
    metrics_df.to_csv(path, index=False)
    print(title)
    print('=' * 70)
    display(metrics_df)
    print('=' * 70)
    return metrics_df


def plot_scatter_performance(y_train, train_predictions, y_test, test_predictions,
                             train_r2, train_rmse, test_r2, test_rmse,
                             title, path):
    fig, ax = plt.subplots(figsize=(DOUBLE_COL_WIDTH, DOUBLE_COL_WIDTH * 0.7))
    ax.scatter(y_train, train_predictions,
               color='royalblue', alpha=0.5, s=15,
               label=f'Train $R^2$={train_r2:.2f}, RMSE={train_rmse:.2f}, N={len(y_train):,}')
    ax.scatter(y_test, test_predictions,
               color='darkorange', alpha=0.5, s=15,
               label=f'Test $R^2$={test_r2:.2f}, RMSE={test_rmse:.2f}, N={len(y_test):,}')
    lower = min(np.min(y_train), np.min(y_test))
    upper = max(np.max(y_train), np.max(y_test))
    ax.plot([lower, upper], [lower, upper], linestyle='--', color='black', linewidth=1, label='1:1 line')
    ax.set_xlabel('True Life Expectancy (years)', fontsize=FONT_SIZE)
    ax.set_ylabel('Predicted Life Expectancy (years)', fontsize=FONT_SIZE)
    ax.set_title(title, fontsize=FONT_SIZE, fontweight='bold')
    ax.legend(fontsize=10, frameon=True, loc='lower right')
    ax.grid(axis='both', linewidth=0.15, alpha=0.3)
    plt.tight_layout()
    plt.savefig(path, dpi=DPI, bbox_inches='tight')
    plt.show()


def plot_qq_comparison(y_train, train_predictions, y_test, test_predictions, title, path):
    train_preds_sorted = np.sort(train_predictions)
    train_targets_sorted = np.sort(y_train)
    test_preds_sorted = np.sort(test_predictions)
    test_targets_sorted = np.sort(y_test)
    all_predictions = np.concatenate((train_predictions, test_predictions))
    all_targets = np.concatenate((np.asarray(y_train), np.asarray(y_test)))

    fig, ax = plt.subplots(figsize=(DOUBLE_COL_WIDTH, DOUBLE_COL_WIDTH * 0.7))
    ax.plot(train_targets_sorted, train_preds_sorted, 'x', label='Train', alpha=0.5, color='royalblue', markersize=3)
    ax.plot(test_targets_sorted, test_preds_sorted, 'o', label='Test', alpha=0.5, color='darkorange', markersize=3)

    lower = min(np.min(all_targets), np.min(all_predictions))
    upper = max(np.max(all_targets), np.max(all_predictions))
    ax.plot([lower, upper], [lower, upper], 'r', linewidth=1, label='Perfect agreement')

    for percentile, color, label in [(25, 'orange', '25th'), (50, 'violet', '50th'),
                                     (75, 'cyan', '75th'), (90, 'green', '90th')]:
        target_pct = np.percentile(np.sort(all_targets), percentile)
        pred_pct = np.percentile(np.sort(all_predictions), percentile)
        ax.plot(target_pct, pred_pct, marker='D', markersize=4,
                color=color, linestyle='None', label=f'{label} percentile')

    ax.set_title(title, fontsize=FONT_SIZE, fontweight='bold')
    ax.set_xlabel('True Life Expectancy (years)', fontsize=FONT_SIZE)
    ax.set_ylabel('Predicted Life Expectancy (years)', fontsize=FONT_SIZE)
    ax.grid(axis='both', linewidth=0.15, alpha=0.3)
    ax.legend(fontsize=8, frameon=True, loc='lower right')
    plt.tight_layout()
    plt.savefig(path, dpi=DPI, bbox_inches='tight')
    plt.show()


def plot_permutation_importance(model, X_test, y_test, output_csv_path, output_fig_path,
                                title, section_name, max_display=20):
    print(f'Calculating permutation importance: {section_name}')
    perm = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42)
    perm_df = pd.DataFrame({
        'Feature': X_test.columns,
        'Display Feature': clean_display_labels(X_test.columns),
        'Importance': perm.importances_mean
    }).sort_values(by='Importance', ascending=False)
    perm_df = perm_df.loc[~perm_df['Feature'].isin(REPORTING_EXCLUDE_FEATURES)].reset_index(drop=True)
    perm_df.to_csv(output_csv_path, index=False)

    fig, ax = plt.subplots(figsize=(DOUBLE_COL_WIDTH, 4.5))
    sns.barplot(x='Importance', y='Display Feature', data=perm_df.head(max_display), palette='viridis', ax=ax)
    ax.set_title(title, fontsize=FONT_SIZE, fontweight='bold')
    ax.set_xlabel('Importance', fontsize=FONT_SIZE)
    ax.set_ylabel('Feature', fontsize=FONT_SIZE)
    plt.tight_layout()
    plt.savefig(output_fig_path, dpi=DPI, bbox_inches='tight')
    plt.show()
    return perm_df


def compute_shap_outputs(model, X_eval, ranking_csv_path, values_path,
                         summary_fig_path=None, bar_fig_path=None,
                         summary_title=None, bar_title=None, max_display=20,
                         summary_figsize=(DOUBLE_COL_WIDTH, 5), bar_figsize=(SINGLE_COL_WIDTH, 4)):
    print('Creating SHAP explainer...')
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_eval)
    if isinstance(shap_values, list):
        shap_values = shap_values[0]
    shap_values = np.array(shap_values)
    np.save(values_path, shap_values)

    ranking_df = pd.DataFrame({
        'Feature': X_eval.columns,
        'Display Feature': clean_display_labels(X_eval.columns),
        'Mean |SHAP|': np.abs(shap_values).mean(axis=0)
    }).sort_values(by='Mean |SHAP|', ascending=False).reset_index(drop=True)
    ranking_df = ranking_df.loc[~ranking_df['Feature'].isin(REPORTING_EXCLUDE_FEATURES)].reset_index(drop=True)
    ranking_df['Rank'] = np.arange(1, len(ranking_df) + 1)
    ranking_df.to_csv(ranking_csv_path, index=False)

    reporting_features = filter_reporting_features(X_eval.columns)
    reporting_indices = [list(X_eval.columns).index(feature) for feature in reporting_features]
    shap_values_display = shap_values[:, reporting_indices]

    X_display = X_eval[reporting_features].copy()
    X_display.columns = clean_display_labels(reporting_features)

    if summary_fig_path is not None:
        plt.figure(figsize=summary_figsize)
        shap.summary_plot(shap_values_display, X_display, max_display=max_display, show=False)
        if summary_title is not None:
            plt.title(summary_title, fontsize=FONT_SIZE, fontweight='bold', pad=10)
        plt.xlabel('SHAP Value (Impact on Model Output)', fontsize=FONT_SIZE)
        plt.tight_layout()
        plt.savefig(summary_fig_path, dpi=DPI, bbox_inches='tight')
        plt.show()

    if bar_fig_path is not None:
        plt.figure(figsize=bar_figsize)
        shap.summary_plot(shap_values_display, X_display, plot_type='bar', max_display=max_display, show=False)
        if bar_title is not None:
            plt.title(bar_title, fontsize=FONT_SIZE, fontweight='bold', pad=10)
        plt.xlabel('Mean |SHAP Value|', fontsize=FONT_SIZE)
        plt.tight_layout()
        plt.savefig(bar_fig_path, dpi=DPI, bbox_inches='tight')
        plt.show()

    return shap_values, ranking_df


print('=' * 70)
print('NOTEBOOK CONFIGURATION READY')
print('=' * 70)
print(f'Base data path: {BASE_DATA_PATH}')
print(f'Smoking data path: {SMOKING_DATA_PATH}')
print(f'County shapefile path: {COUNTY_SHP_PATH}')
print(f'Revision output directory: {OUTPUT_DIR}')

NOTEBOOK CONFIGURATION READY
Base data path: ../data_cleaned/combined_final/final_combined_all_variables_reduced.csv
Smoking data path: /Users/samyakshrestha/Projects/predicting-lung-cancer/data/processed/smoking_by_year/smoking_all_years.csv
County shapefile path: ../data/shapefiles/cb_2019_us_county_20m.shp
Revision output directory: ../data_cleaned/outputs_cleaned/modeling/xgboost/revision


---
## Section 1: Load Base Data and Merge Smoking

This section creates the smoking-augmented revision dataset. It preserves the full county-year panel first, then builds the complete-case dataset used for the primary revised model.


### 1.1 Load Base Reduced Dataset and Smoking File

Load the county-year life expectancy panel and the smoking covariate file before any merges or filtering.


In [ ]:
# Load base reduced dataset and smoking data
print('=' * 70)
print('LOADING DATASETS')
print('=' * 70)

df_base = pd.read_csv(BASE_DATA_PATH)
smoking_df = pd.read_csv(SMOKING_DATA_PATH)

print(f'Base reduced dataset shape: {df_base.shape}')
print(f'Smoking dataset shape: {smoking_df.shape}')
print(f'Base years: {sorted(df_base["Year"].unique())}')
print(f'Smoking years: {sorted(smoking_df["Year"].unique())}')


### 1.2 Merge Smoking Data and Build the Complete-Case Model B Dataset

Merge smoking onto the base panel, rename the smoking feature for presentation consistency, and construct the complete-case dataset used for the canonical revised model.


In [ ]:
# Merge smoking data onto the base LE panel
smoking_cols = ['fips', 'Year', 'smoking_rate', 'is_post_2015']
df_model_b_full = df_base.merge(
    smoking_df[smoking_cols],
    how='left',
    left_on=['Fips', 'Year'],
    right_on=['fips', 'Year']
)

if 'fips' in df_model_b_full.columns:
    df_model_b_full = df_model_b_full.drop(columns=['fips'])

# Complete-case primary dataset
df_model_b_full = df_model_b_full.rename(columns={'smoking_rate': SMOKING_FEATURE})

missing_smoking = int(df_model_b_full[SMOKING_FEATURE].isna().sum())
complete_rows = int(df_model_b_full[SMOKING_FEATURE].notna().sum())
sample_loss_fraction = missing_smoking / len(df_model_b_full)

df_model_b_complete = df_model_b_full.dropna(subset=[SMOKING_FEATURE]).copy().reset_index(drop=True)

# Confirm feature count
X_model_b_complete, y_model_b_complete, groups_model_b_complete = prepare_xy(df_model_b_complete)
assert X_model_b_complete.shape[1] == EXPECTED_MODEL_B_FEATURES, (
    f'Expected {EXPECTED_MODEL_B_FEATURES} features, found {X_model_b_complete.shape[1]}'
)


### 1.3 Merge Summary and Feature Inventory

Confirm row retention, missingness, county coverage, and the final Model B feature list after the smoking merge.


In [ ]:
print('\n' + '=' * 70)
print('MODEL B DATASET SUMMARY')
print('=' * 70)
print(f'Rows before smoking merge filter: {len(df_model_b_full):,}')
print(f'Missing smoking values: {missing_smoking:,}')
print(f'Complete-case rows retained: {complete_rows:,}')
print(f'Sample loss fraction: {sample_loss_fraction:.3%}')
print(f'Complete-case unique counties: {groups_model_b_complete.nunique():,}')
print(f'Number of Model B features: {X_model_b_complete.shape[1]}')
print('\nFeature list (Model B complete-case):')
for i, col in enumerate(X_model_b_complete.columns, 1):
    print(f'  {i:2}. {clean_display_labels(col)}')

merge_summary = {
    'base_rows': int(len(df_base)),
    'merged_rows': int(len(df_model_b_full)),
    'missing_smoking': missing_smoking,
    'complete_case_rows': complete_rows,
    'sample_loss_fraction': sample_loss_fraction,
    'complete_case_unique_counties': int(groups_model_b_complete.nunique()),
    'model_b_feature_count': int(X_model_b_complete.shape[1])
}
save_json(merge_summary, OUTPUT_DIR / 'model_b_merge_summary.json')
record_artifact(OUTPUT_DIR / 'model_b_merge_summary.json', 'Section 1', 'Smoking merge summary', False)


---
## Section 2: Model B Main Run (Complete-Case First)

This is the primary revised model. It follows the Run4 backbone exactly: group-wise county split, BayesSearchCV, permutation importance, SHAP analysis, and publication-style metrics/figures.


### 2.1 Train/Test Split and Leakage Check

Replicate the Run4 county-level split using `GroupShuffleSplit`, then verify that no county appears in both the training and test partitions.


In [ ]:
# Prepare Model B train/test split
X_model_b, y_model_b, groups_model_b = prepare_xy(df_model_b_complete)

gss_model_b = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx_model_b, test_idx_model_b = next(gss_model_b.split(X_model_b, y_model_b, groups=groups_model_b))

X_train_model_b = X_model_b.iloc[train_idx_model_b].copy()
X_test_model_b = X_model_b.iloc[test_idx_model_b].copy()
y_train_model_b = y_model_b.iloc[train_idx_model_b].copy()
y_test_model_b = y_model_b.iloc[test_idx_model_b].copy()
groups_train_model_b = groups_model_b.iloc[train_idx_model_b].copy()

test_counties = set(groups_model_b.iloc[test_idx_model_b])
train_counties = set(groups_model_b.iloc[train_idx_model_b])
county_overlap = train_counties.intersection(test_counties)

print('=' * 70)
print('MODEL B TRAIN/TEST SPLIT')
print('=' * 70)
print(f'Training rows: {len(X_train_model_b):,}')
print(f'Test rows: {len(X_test_model_b):,}')
print(f'Training counties: {len(train_counties):,}')
print(f'Test counties: {len(test_counties):,}')
print(f'County overlap: {len(county_overlap)}')
print(f'Leakage status: {"NO LEAKAGE" if len(county_overlap) == 0 else "LEAKAGE DETECTED"}')

split_metadata = {
    'train_rows': int(len(X_train_model_b)),
    'test_rows': int(len(X_test_model_b)),
    'train_counties': int(len(train_counties)),
    'test_counties': int(len(test_counties)),
    'county_overlap': int(len(county_overlap)),
    'random_state': 42,
    'test_size': 0.2,
}
save_json(split_metadata, OUTPUT_DIR / 'model_b_split_metadata.json')
record_artifact(OUTPUT_DIR / 'model_b_split_metadata.json', 'Section 2', 'Model B split metadata', False)


### 2.2 Bayesian Hyperparameter Optimization

Run the 50-iteration `BayesSearchCV` procedure on the Model B training data and save the full search history and best parameter set.


In [ ]:
# Run Bayesian optimization for Model B
bayes_model_b, best_model_b = run_bayes_xgb(
    X_train_model_b,
    y_train_model_b,
    groups_train_model_b,
    section_name='Model B (45 features, complete-case)'
)

best_params_model_b = _json_safe(bayes_model_b.best_params_)
save_json(best_params_model_b, OUTPUT_DIR / 'model_b_best_params.json')
record_artifact(OUTPUT_DIR / 'model_b_best_params.json', 'Section 2', 'Best hyperparameters for Model B', False)

cv_results_model_b = pd.DataFrame(bayes_model_b.cv_results_)
cv_results_model_b.to_csv(OUTPUT_DIR / 'model_b_cv_results.csv', index=False)
record_artifact(OUTPUT_DIR / 'model_b_cv_results.csv', 'Section 2', 'Raw BayesSearchCV results for Model B', False)


### 2.3 Model Training and Predictions

Use the selected Model B estimator to generate training and held-out predictions for all downstream analyses.


In [ ]:
# Predictions and metrics
train_predictions_model_b = best_model_b.predict(X_train_model_b)
test_predictions_model_b = best_model_b.predict(X_test_model_b)
metrics_model_b = evaluate_predictions(
    y_train_model_b,
    train_predictions_model_b,
    y_test_model_b,
    test_predictions_model_b,
    n_features=X_train_model_b.shape[1]
)


### 2.4 Model Evaluation and Prediction Tables

Print the headline performance metrics, save the metrics table, and export train/test prediction tables with residuals.


In [ ]:
print('\n' + '=' * 70)
print('MODEL B PERFORMANCE METRICS')
print('=' * 70)
print(f"Training R²:   {metrics_model_b['train_r2']:.4f}")
print(f"Training RMSE: {metrics_model_b['train_rmse']:.2f} years")
print(f"Training MAE:  {metrics_model_b['train_mae']:.2f} years")
print(f"Test R²:       {metrics_model_b['test_r2']:.4f}")
print(f"Test RMSE:     {metrics_model_b['test_rmse']:.2f} years")
print(f"Test MAE:      {metrics_model_b['test_mae']:.2f} years")

metrics_model_b_df = save_metrics_table(
    metrics_model_b,
    OUTPUT_DIR / 'model_b_metrics.csv',
    title='Model B Metrics Table'
)
record_artifact(OUTPUT_DIR / 'model_b_metrics.csv', 'Section 2', 'Model B metrics table', True)

# Save prediction-level outputs for downstream analyses
model_b_test_predictions = df_model_b_complete.loc[test_idx_model_b, IDENTIFIER_COLS + [TARGET_COL, 'Poverty Rate', "Bachelor's Degree or Higher (%)", SMOKING_FEATURE, FORMALDEHYDE_FEATURE]].copy()
model_b_test_predictions['Predicted Life Expectancy'] = test_predictions_model_b
model_b_test_predictions['Residual'] = model_b_test_predictions[TARGET_COL] - model_b_test_predictions['Predicted Life Expectancy']
model_b_test_predictions.to_csv(OUTPUT_DIR / 'model_b_test_predictions.csv', index=False)
record_artifact(OUTPUT_DIR / 'model_b_test_predictions.csv', 'Section 2', 'Model B test predictions and residuals', False)

model_b_train_predictions = df_model_b_complete.loc[train_idx_model_b, IDENTIFIER_COLS + [TARGET_COL]].copy()
model_b_train_predictions['Predicted Life Expectancy'] = train_predictions_model_b
model_b_train_predictions['Residual'] = model_b_train_predictions[TARGET_COL] - model_b_train_predictions['Predicted Life Expectancy']
model_b_train_predictions.to_csv(OUTPUT_DIR / 'model_b_train_predictions.csv', index=False)
record_artifact(OUTPUT_DIR / 'model_b_train_predictions.csv', 'Section 2', 'Model B train predictions and residuals', False)


### 2.5 Scatter Plot: Predictions vs. Actual

Create the paper-facing Model B performance scatter plot.


In [ ]:
plot_scatter_performance(
    y_train_model_b, train_predictions_model_b,
    y_test_model_b, test_predictions_model_b,
    metrics_model_b['train_r2'], metrics_model_b['train_rmse'],
    metrics_model_b['test_r2'], metrics_model_b['test_rmse'],
    title='Model Performance: Predictions vs. Actual (Model B)',
    path=OUTPUT_DIR / 'model_b_scatter_performance.png'
)
record_artifact(OUTPUT_DIR / 'model_b_scatter_performance.png', 'Section 2', 'Model B scatter performance figure', True)


### 2.6 Q-Q Plot

Create the internal Q-Q diagnostic plot for the Model B prediction residual structure.


In [ ]:
plot_qq_comparison(
    y_train_model_b, train_predictions_model_b,
    y_test_model_b, test_predictions_model_b,
    title='Q-Q Plot: Residuals Normality Check (Model B)',
    path=OUTPUT_DIR / 'model_b_qq_plot.png'
)
record_artifact(OUTPUT_DIR / 'model_b_qq_plot.png', 'Section 2', 'Model B Q-Q plot', False)


### 2.7 Permutation Importance

Compute and save permutation importance for the held-out Model B test set, excluding `is_post_2015` from reported rankings.


In [ ]:
perm_importance_model_b = plot_permutation_importance(
    best_model_b,
    X_test_model_b,
    y_test_model_b,
    output_csv_path=OUTPUT_DIR / 'model_b_permutation_importance.csv',
    output_fig_path=OUTPUT_DIR / 'model_b_permutation_importance.png',
    title='Permutation Importance (Top 25 Features, Model B)',
    section_name='Model B',
    max_display=25
)
record_artifact(OUTPUT_DIR / 'model_b_permutation_importance.csv', 'Section 2', 'Model B permutation importance table', False)
record_artifact(OUTPUT_DIR / 'model_b_permutation_importance.png', 'Section 2', 'Model B permutation importance figure', True)


### 2.8 SHAP Analysis

Compute SHAP values, save the ranking table and raw SHAP array, and generate the paper-facing SHAP beeswarm and bar plots.


In [ ]:
shap_values_model_b, shap_ranking_model_b = compute_shap_outputs(
    best_model_b,
    X_test_model_b,
    ranking_csv_path=OUTPUT_DIR / 'model_b_shap_ranking.csv',
    values_path=OUTPUT_DIR / 'model_b_shap_values.npy',
    summary_fig_path=OUTPUT_DIR / 'model_b_shap_summary.png',
    bar_fig_path=OUTPUT_DIR / 'model_b_shap_bar.png',
    summary_title='SHAP Summary Plot (Model B, Top 20)',
    bar_title='SHAP Feature Importance (Model B, Top 20)',
    max_display=20,
    summary_figsize=(DOUBLE_COL_WIDTH, 5),
    bar_figsize=(SINGLE_COL_WIDTH, 4)
)
record_artifact(OUTPUT_DIR / 'model_b_shap_ranking.csv', 'Section 2', 'Model B SHAP ranking table', False)
record_artifact(OUTPUT_DIR / 'model_b_shap_values.npy', 'Section 2', 'Raw Model B SHAP values', False)
record_artifact(OUTPUT_DIR / 'model_b_shap_summary.png', 'Section 2', 'Model B SHAP beeswarm plot', True)
record_artifact(OUTPUT_DIR / 'model_b_shap_bar.png', 'Section 2', 'Model B SHAP bar plot', True)

model_b_formaldehyde_rank = int(
    shap_ranking_model_b.loc[shap_ranking_model_b['Feature'] == FORMALDEHYDE_FEATURE, 'Rank'].iloc[0]
)
print(f'\nFormaldehyde rank in Model B (mean |SHAP|): {model_b_formaldehyde_rank}')


---
## Section 3: Explicit Decision Gate for Smoking Imputation

This section keeps the smoking-imputation branch conditional. The primary revised model remains the complete-case Model B, but an imputed sensitivity run will be triggered if the complete-case result looks materially unstable.


### 3.1 Decision Gate Summary

Evaluate whether the complete-case Model B result is stable enough to remain canonical without an imputed smoking sensitivity rerun.


In [ ]:
# Decision gate for optional smoking imputation sensitivity analysis
r2_drop_vs_run4 = RUN4_BASELINE_TEST_R2 - metrics_model_b['test_r2']
run_imputation_sensitivity = any([
    sample_loss_fraction > 0.10,
    r2_drop_vs_run4 > 0.02,
    model_b_formaldehyde_rank > 5,
])

decision_gate_summary = {
    'sample_loss_fraction': sample_loss_fraction,
    'r2_drop_vs_run4': r2_drop_vs_run4,
    'formaldehyde_rank_model_b': model_b_formaldehyde_rank,
    'trigger_sample_loss_gt_10pct': bool(sample_loss_fraction > 0.10),
    'trigger_r2_drop_gt_0_02': bool(r2_drop_vs_run4 > 0.02),
    'trigger_formaldehyde_rank_gt_5': bool(model_b_formaldehyde_rank > 5),
    'run_imputation_sensitivity': bool(run_imputation_sensitivity),
    'canonical_revised_model': 'complete_case_model_b'
}

save_json(decision_gate_summary, OUTPUT_DIR / 'model_b_smoking_gate_summary.json')
record_artifact(OUTPUT_DIR / 'model_b_smoking_gate_summary.json', 'Section 3', 'Smoking sensitivity decision gate summary', False)

print('=' * 70)
print('SMOKING SENSITIVITY DECISION GATE')
print('=' * 70)
print(f'Sample loss fraction: {sample_loss_fraction:.3%}')
print(f'R² drop vs Run4 baseline: {r2_drop_vs_run4:.4f}')
print(f'Formaldehyde SHAP rank: {model_b_formaldehyde_rank}')
print(f'Run imputation sensitivity: {run_imputation_sensitivity}')

imputed_comparison_df = None


### 3.2 Conditional Smoking-Imputed Sensitivity Run

Only if the decision gate is triggered, fit a state-year-median-imputed smoking model and compare it against the canonical complete-case Model B.


In [ ]:
if run_imputation_sensitivity:
    print('\nRunning imputed smoking sensitivity analysis...')

    df_model_b_imputed = df_model_b_full.copy()
    state_year_median = df_model_b_imputed.groupby(['State', 'Year'])[SMOKING_FEATURE].transform('median')
    df_model_b_imputed[SMOKING_FEATURE] = df_model_b_imputed[SMOKING_FEATURE].fillna(state_year_median)

    remaining_missing = int(df_model_b_imputed[SMOKING_FEATURE].isna().sum())
    if remaining_missing > 0:
        raise ValueError(
            f'State-year median imputation left {remaining_missing} missing smoking values. '\
            'Notebook plan assumes state-year imputation is sufficient.'
        )

    df_model_b_imputed = df_model_b_imputed.reset_index(drop=True)
    X_imputed, y_imputed, groups_imputed = prepare_xy(df_model_b_imputed)

    gss_imputed = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx_imputed, test_idx_imputed = next(gss_imputed.split(X_imputed, y_imputed, groups=groups_imputed))

    X_train_imputed = X_imputed.iloc[train_idx_imputed].copy()
    X_test_imputed = X_imputed.iloc[test_idx_imputed].copy()
    y_train_imputed = y_imputed.iloc[train_idx_imputed].copy()
    y_test_imputed = y_imputed.iloc[test_idx_imputed].copy()
    groups_train_imputed = groups_imputed.iloc[train_idx_imputed].copy()

    bayes_imputed, best_model_imputed = run_bayes_xgb(
        X_train_imputed,
        y_train_imputed,
        groups_train_imputed,
        section_name='Smoking-imputed sensitivity model'
    )

    train_pred_imputed = best_model_imputed.predict(X_train_imputed)
    test_pred_imputed = best_model_imputed.predict(X_test_imputed)
    metrics_imputed = evaluate_predictions(
        y_train_imputed,
        train_pred_imputed,
        y_test_imputed,
        test_pred_imputed,
        n_features=X_train_imputed.shape[1]
    )

    _, shap_ranking_imputed = compute_shap_outputs(
        best_model_imputed,
        X_test_imputed,
        ranking_csv_path=OUTPUT_DIR / 'model_b_imputed_shap_ranking.csv',
        values_path=OUTPUT_DIR / 'model_b_imputed_shap_values.npy',
        summary_fig_path=None,
        bar_fig_path=None,
        max_display=20
    )

    formaldehyde_rank_imputed = int(
        shap_ranking_imputed.loc[shap_ranking_imputed['Feature'] == FORMALDEHYDE_FEATURE, 'Rank'].iloc[0]
    )

    imputed_comparison_df = pd.DataFrame([
        {
            'Model': 'Complete-case Model B',
            'Rows': len(df_model_b_complete),
            'Test R²': metrics_model_b['test_r2'],
            'Test RMSE': metrics_model_b['test_rmse'],
            'Test MAE': metrics_model_b['test_mae'],
            'Formaldehyde Rank': model_b_formaldehyde_rank,
        },
        {
            'Model': 'Smoking-imputed sensitivity',
            'Rows': len(df_model_b_imputed),
            'Test R²': metrics_imputed['test_r2'],
            'Test RMSE': metrics_imputed['test_rmse'],
            'Test MAE': metrics_imputed['test_mae'],
            'Formaldehyde Rank': formaldehyde_rank_imputed,
        }
    ])

    imputed_comparison_df.to_csv(OUTPUT_DIR / 'model_b_imputed_comparison.csv', index=False)
    save_json(_json_safe(bayes_imputed.best_params_), OUTPUT_DIR / 'model_b_imputed_best_params.json')
    record_artifact(OUTPUT_DIR / 'model_b_imputed_comparison.csv', 'Section 3', 'Smoking-imputed sensitivity comparison', False)
    record_artifact(OUTPUT_DIR / 'model_b_imputed_best_params.json', 'Section 3', 'Smoking-imputed sensitivity best parameters', False)

    print('\nSmoking-imputed sensitivity comparison:')
    display(imputed_comparison_df)
else:
    print('\nNo imputation sensitivity run is required. Complete-case Model B remains the canonical revised model.')


---
## Section 4: Formaldehyde vs. Socioeconomic Confounding

This section addresses Reviewer 2 Major Comment 1 using a visual and numeric pair:
1. poverty-quartile SHAP analysis for formaldehyde
2. partial correlation after residualizing on poverty, education, and smoking


### 4.1 Build the Formaldehyde Confounding Analysis Table

Assemble the held-out Model B test rows with formaldehyde exposure, poverty, education, smoking, and the corresponding SHAP values for formaldehyde.


In [ ]:
# Build confounding analysis table from the canonical complete-case Model B test set
formaldehyde_index = list(X_test_model_b.columns).index(FORMALDEHYDE_FEATURE)

confounding_df = df_model_b_complete.loc[test_idx_model_b, IDENTIFIER_COLS + [TARGET_COL, 'Poverty Rate', "Bachelor's Degree or Higher (%)", SMOKING_FEATURE, FORMALDEHYDE_FEATURE]].copy()
confounding_df = confounding_df.reset_index(drop=True)
confounding_df['Formaldehyde SHAP Value'] = shap_values_model_b[:, formaldehyde_index]


### 4.2 Poverty-Quartile SHAP Figure

Stratify the held-out test set into poverty quartiles and visualize how the formaldehyde SHAP relationship behaves within each quartile.


In [ ]:
# Create poverty quartiles using rank-first to guarantee four equally sized groups
poverty_labels = ['Q1: Lowest Poverty', 'Q2', 'Q3', 'Q4: Highest Poverty']
confounding_df['Poverty Quartile'] = pd.qcut(
    confounding_df['Poverty Rate'].rank(method='first'),
    q=4,
    labels=poverty_labels
)

x_min = confounding_df[FORMALDEHYDE_FEATURE].min()
x_max = confounding_df[FORMALDEHYDE_FEATURE].max()
y_min = confounding_df['Formaldehyde SHAP Value'].min()
y_max = confounding_df['Formaldehyde SHAP Value'].max()

fig, axes = plt.subplots(2, 2, figsize=(DOUBLE_COL_WIDTH * 1.25, DOUBLE_COL_WIDTH * 1.0), sharex=True, sharey=True)
axes = axes.flatten()

for ax, quartile in zip(axes, poverty_labels):
    subset = confounding_df[confounding_df['Poverty Quartile'] == quartile]
    ax.scatter(subset[FORMALDEHYDE_FEATURE], subset['Formaldehyde SHAP Value'],
               alpha=0.55, s=16, color='steelblue')
    ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
    ax.set_title(quartile, fontsize=FONT_SIZE, fontweight='bold')
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.grid(axis='both', linewidth=0.15, alpha=0.3)

for ax in axes[2:]:
    ax.set_xlabel(clean_display_labels(FORMALDEHYDE_FEATURE), fontsize=FONT_SIZE - 1)
for ax in [axes[0], axes[2]]:
    ax.set_ylabel('Formaldehyde SHAP Value', fontsize=FONT_SIZE - 1)

plt.suptitle('Formaldehyde SHAP Values Stratified by Poverty Quartile', fontsize=FONT_SIZE + 1, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confounding_formaldehyde_poverty_quartiles.png', dpi=DPI, bbox_inches='tight')
plt.show()
record_artifact(OUTPUT_DIR / 'confounding_formaldehyde_poverty_quartiles.png', 'Section 4', 'Poverty-quartile formaldehyde SHAP figure', True)


### 4.3 Partial Correlation After Residualization

Residualize formaldehyde exposure and life expectancy on poverty, education, and smoking, then compute the partial correlation between the residualized variables.


In [ ]:
# Numeric confounding check via partial correlation after residualization
control_vars = ['Poverty Rate', "Bachelor's Degree or Higher (%)", SMOKING_FEATURE]
controls = sm.add_constant(confounding_df[control_vars])
resid_formaldehyde = sm.OLS(confounding_df[FORMALDEHYDE_FEATURE], controls).fit().resid
resid_life_expectancy = sm.OLS(confounding_df[TARGET_COL], controls).fit().resid
partial_corr, partial_p = pearsonr(resid_formaldehyde, resid_life_expectancy)

partial_corr_df = pd.DataFrame([
    {
        'Exposure Variable': FORMALDEHYDE_FEATURE,
        'Outcome Variable': TARGET_COL,
        'Controls': '; '.join(control_vars),
        'Partial Correlation': partial_corr,
        'P-value': partial_p,
        'N': len(confounding_df)
    }
])
partial_corr_df.to_csv(OUTPUT_DIR / 'confounding_partial_correlation.csv', index=False)
record_artifact(OUTPUT_DIR / 'confounding_partial_correlation.csv', 'Section 4', 'Partial correlation table controlling for SES and smoking', False)

print('=' * 70)
print('FORMALDEHYDE CONFOUNDING ANALYSIS')
print('=' * 70)
print(f'Partial correlation (formaldehyde vs life expectancy | poverty, education, smoking): {partial_corr:.4f}')
print(f'P-value: {partial_p:.4e}')
display(partial_corr_df)


---
## Section 5: Temporal Validation

This section tests whether the revised model generalizes across time, not just across counties. It trains on 2012–2016 and evaluates on 2017–2019 using the best hyperparameters from the canonical Model B run.


### 5.1 Temporal Train/Test Partition

Build the 2012–2016 training set and 2017–2019 test set using the same Model B feature schema.


In [ ]:
# Temporal validation using Model B best hyperparameters
train_years = list(range(2012, 2017))
test_years = list(range(2017, 2020))

df_temporal_train = df_model_b_complete[df_model_b_complete['Year'].isin(train_years)].copy().reset_index(drop=True)
df_temporal_test = df_model_b_complete[df_model_b_complete['Year'].isin(test_years)].copy().reset_index(drop=True)

X_temporal_train, y_temporal_train, _ = prepare_xy(df_temporal_train)
X_temporal_test, y_temporal_test, _ = prepare_xy(df_temporal_test)


### 5.2 Fixed-Parameter Temporal Model

Refit the revised model on the temporal training period using the best hyperparameters from the canonical Model B run.


In [ ]:
temporal_model = build_fixed_param_model(best_params_model_b)
temporal_model.fit(X_temporal_train, y_temporal_train)

temporal_train_predictions = temporal_model.predict(X_temporal_train)
temporal_test_predictions = temporal_model.predict(X_temporal_test)
temporal_metrics = evaluate_predictions(
    y_temporal_train,
    temporal_train_predictions,
    y_temporal_test,
    temporal_test_predictions,
    n_features=X_temporal_train.shape[1]
)

temporal_metrics_df = save_metrics_table(
    temporal_metrics,
    OUTPUT_DIR / 'temporal_metrics.csv',
    title='Temporal Validation Metrics'
)
record_artifact(OUTPUT_DIR / 'temporal_metrics.csv', 'Section 5', 'Temporal validation metrics table', True)


### 5.3 Temporal Outputs

Save the temporal prediction table, metrics table, and the temporal performance scatter plot.


In [ ]:
temporal_predictions_df = df_temporal_test[IDENTIFIER_COLS + [TARGET_COL]].copy()
temporal_predictions_df['Predicted Life Expectancy'] = temporal_test_predictions
temporal_predictions_df['Residual'] = temporal_predictions_df[TARGET_COL] - temporal_predictions_df['Predicted Life Expectancy']
temporal_predictions_df.to_csv(OUTPUT_DIR / 'temporal_test_predictions.csv', index=False)
record_artifact(OUTPUT_DIR / 'temporal_test_predictions.csv', 'Section 5', 'Temporal validation prediction table', False)

plot_scatter_performance(
    y_temporal_train, temporal_train_predictions,
    y_temporal_test, temporal_test_predictions,
    temporal_metrics['train_r2'], temporal_metrics['train_rmse'],
    temporal_metrics['test_r2'], temporal_metrics['test_rmse'],
    title='Temporal Validation: Predictions vs. Actual',
    path=OUTPUT_DIR / 'temporal_scatter_performance.png'
)
record_artifact(OUTPUT_DIR / 'temporal_scatter_performance.png', 'Section 5', 'Temporal validation scatter figure', True)


---
## Section 6: Moran’s I Spatial Residual Check

This section tests whether Model B residuals are spatially clustered after aggregating test residuals to the county level. This avoids the incorrect county-year-row Moran’s I formulation.


### 6.1 Prepare County Geometries and County-Level Mean Residuals

Load the county shapefile, harmonize FIPS identifiers, and aggregate Model B test residuals to one mean residual per held-out county.


In [ ]:
# Moran's I on county-level mean residuals for held-out Model B test counties
print('=' * 70)
print("MORAN'S I SPATIAL RESIDUAL CHECK")
print('=' * 70)

spatial_gdf = gpd.read_file(COUNTY_SHP_PATH)

if 'GEOID' in spatial_gdf.columns:
    spatial_gdf['FIPS'] = spatial_gdf['GEOID'].astype(str).str.zfill(5)
elif {'STATEFP', 'COUNTYFP'}.issubset(spatial_gdf.columns):
    spatial_gdf['FIPS'] = spatial_gdf['STATEFP'].astype(str).str.zfill(2) + spatial_gdf['COUNTYFP'].astype(str).str.zfill(3)
else:
    raise ValueError('Could not determine county FIPS column from shapefile.')

study_counties = set(df_base['Fips'].astype(str).str.zfill(5))
spatial_gdf = spatial_gdf[spatial_gdf['FIPS'].isin(study_counties)].copy()
spatial_gdf['FIPS'] = spatial_gdf['FIPS'].astype(str).str.zfill(5)

county_residuals = model_b_test_predictions.copy()
county_residuals['FIPS'] = county_residuals['Fips'].astype(str).str.zfill(5)
county_residuals_mean = county_residuals.groupby('FIPS', as_index=False)['Residual'].mean()
county_residuals_mean = county_residuals_mean.rename(columns={'Residual': 'Mean Residual'})

spatial_residuals_gdf = spatial_gdf.merge(county_residuals_mean, on='FIPS', how='inner').copy()
spatial_residuals_gdf = spatial_residuals_gdf[['FIPS', 'geometry', 'Mean Residual']].dropna().copy()


### 6.2 Moran's I Summary Table

Construct Queen contiguity weights, compute Moran's I on county-level mean residuals, and save the summary table.


In [ ]:
weights = Queen.from_dataframe(spatial_residuals_gdf, ids=spatial_residuals_gdf['FIPS'].tolist())
weights.transform = 'r'
moran_result = Moran(spatial_residuals_gdf['Mean Residual'].values, weights)

moran_df = pd.DataFrame([
    {
        'Moran I': moran_result.I,
        'Expected I': moran_result.EI,
        'Z-score': moran_result.z_norm,
        'P-value': moran_result.p_norm,
        'N Counties': len(spatial_residuals_gdf)
    }
])

moran_df.to_csv(OUTPUT_DIR / 'spatial_morans_i.csv', index=False)
record_artifact(OUTPUT_DIR / 'spatial_morans_i.csv', 'Section 6', "Moran's I residual summary", True)


### 6.3 Residual Map

Create an internal map of county-level mean residuals for visual inspection of spatial patterning.


In [ ]:
fig, ax = plt.subplots(figsize=(DOUBLE_COL_WIDTH, DOUBLE_COL_WIDTH * 0.75))
spatial_residuals_gdf.plot(column='Mean Residual', cmap='coolwarm', linewidth=0.1, ax=ax, legend=True)
ax.set_title("County-Level Mean Residuals (Model B Test Counties)", fontsize=FONT_SIZE, fontweight='bold')
ax.set_axis_off()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'spatial_mean_residual_map.png', dpi=DPI, bbox_inches='tight')
plt.show()
record_artifact(OUTPUT_DIR / 'spatial_mean_residual_map.png', 'Section 6', 'County mean residual map', False)

print(moran_df.to_string(index=False))


---
## Section 7: SHAP Stability Across GroupKFold Folds

This section uses the canonical Model B hyperparameters and evaluates formaldehyde’s SHAP rank across 5 held-out county folds.


### 7.1 Fold-Level SHAP Stability Loop

Refit the fixed-parameter Model B across five county-grouped folds and record the formaldehyde rank and top-10 feature list in each fold.


In [ ]:
# SHAP stability across GroupKFold folds using fixed Model B hyperparameters
print('=' * 70)
print('SHAP STABILITY ACROSS GROUPKFOLD FOLDS')
print('=' * 70)

gkf = GroupKFold(n_splits=5)
stability_records = []

for fold_num, (fold_train_idx, fold_val_idx) in enumerate(gkf.split(X_model_b, y_model_b, groups=groups_model_b), start=1):
    print(f'Fold {fold_num}/5')
    X_fold_train = X_model_b.iloc[fold_train_idx].copy()
    X_fold_val = X_model_b.iloc[fold_val_idx].copy()
    y_fold_train = y_model_b.iloc[fold_train_idx].copy()
    y_fold_val = y_model_b.iloc[fold_val_idx].copy()

    fold_model = build_fixed_param_model(best_params_model_b)
    fold_model.fit(X_fold_train, y_fold_train)

    fold_explainer = shap.TreeExplainer(fold_model)
    fold_shap_values = fold_explainer.shap_values(X_fold_val)
    if isinstance(fold_shap_values, list):
        fold_shap_values = fold_shap_values[0]
    fold_shap_values = np.array(fold_shap_values)

    fold_ranking = pd.DataFrame({
        'Feature': X_fold_val.columns,
        'Mean |SHAP|': np.abs(fold_shap_values).mean(axis=0)
    }).sort_values(by='Mean |SHAP|', ascending=False).reset_index(drop=True)
    fold_ranking = fold_ranking.loc[~fold_ranking['Feature'].isin(REPORTING_EXCLUDE_FEATURES)].reset_index(drop=True)
    fold_ranking['Rank'] = np.arange(1, len(fold_ranking) + 1)

    formaldehyde_rank_fold = int(fold_ranking.loc[fold_ranking['Feature'] == FORMALDEHYDE_FEATURE, 'Rank'].iloc[0])
    top_10_fold = fold_ranking['Feature'].head(10).tolist()

    stability_records.append({
        'Fold': fold_num,
        'Validation Rows': len(X_fold_val),
        'Validation Counties': int(groups_model_b.iloc[fold_val_idx].nunique()),
        'Formaldehyde Rank': formaldehyde_rank_fold,
        'Top 10 Features': ' | '.join(clean_display_labels(top_10_fold))
    })


### 7.2 Stability Summary

Save the fold-level stability table and a compact JSON summary for the response letter.


In [ ]:
stability_df = pd.DataFrame(stability_records)
stability_df.to_csv(OUTPUT_DIR / 'stability_shap_ranks.csv', index=False)
record_artifact(OUTPUT_DIR / 'stability_shap_ranks.csv', 'Section 7', 'SHAP rank stability across folds', False)

stability_summary = {
    'mean_formaldehyde_rank': float(stability_df['Formaldehyde Rank'].mean()),
    'median_formaldehyde_rank': float(stability_df['Formaldehyde Rank'].median()),
    'max_formaldehyde_rank': int(stability_df['Formaldehyde Rank'].max()),
    'min_formaldehyde_rank': int(stability_df['Formaldehyde Rank'].min()),
    'times_formaldehyde_top3': int((stability_df['Formaldehyde Rank'] <= 3).sum()),
}
save_json(stability_summary, OUTPUT_DIR / 'stability_shap_summary.json')
record_artifact(OUTPUT_DIR / 'stability_shap_summary.json', 'Section 7', 'SHAP stability summary for response letter', False)

print('SHAP stability table:')
display(stability_df)
print('\nSummary:')
print(stability_summary)


---
## Section 8: Bayesian Optimization Convergence

This section summarizes whether the Model B Bayesian optimization stabilized within the 50-iteration search budget.


### 8.1 Convergence Table

Convert the raw `BayesSearchCV` results into an iteration-by-iteration convergence table with the best score observed so far.


In [ ]:
# Convergence behavior from Model B BayesSearchCV
print('=' * 70)
print('BAYESIAN OPTIMIZATION CONVERGENCE')
print('=' * 70)

convergence_df = cv_results_model_b.copy().reset_index(drop=True)
convergence_df['Iteration'] = np.arange(1, len(convergence_df) + 1)
convergence_df['Best Score So Far'] = convergence_df['mean_test_score'].cummax()
convergence_df.to_csv(OUTPUT_DIR / 'convergence_model_b.csv', index=False)
record_artifact(OUTPUT_DIR / 'convergence_model_b.csv', 'Section 8', 'Model B convergence table', False)


### 8.2 Convergence Plot

Plot the raw fold-mean validation score and the best-so-far trajectory across the 50 optimization iterations.


In [ ]:
fig, ax = plt.subplots(figsize=(DOUBLE_COL_WIDTH, DOUBLE_COL_WIDTH * 0.6))
ax.plot(convergence_df['Iteration'], convergence_df['mean_test_score'], marker='o', linewidth=1.5, label='Fold-mean CV R²')
ax.plot(convergence_df['Iteration'], convergence_df['Best Score So Far'], marker='s', linewidth=2.0, label='Best score so far')
ax.set_xlabel('BayesSearchCV Iteration', fontsize=FONT_SIZE)
ax.set_ylabel('Cross-validated R²', fontsize=FONT_SIZE)
ax.set_title('Bayesian Optimization Convergence (Model B)', fontsize=FONT_SIZE, fontweight='bold')
ax.grid(axis='both', linewidth=0.15, alpha=0.3)
ax.legend(frameon=True)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'convergence_model_b.png', dpi=DPI, bbox_inches='tight')
plt.show()
record_artifact(OUTPUT_DIR / 'convergence_model_b.png', 'Section 8', 'Model B convergence figure', False)


---
## Section 9: CAMS Coarse-Resolution Context Analysis

This section approximates the atmospheric support problem using county centroids assigned to 0.75° x 0.75° support regions. It is explicitly a diagnostic approximation, not a reconstruction of the multilinear interpolation pipeline.


### 9.1 Approximate CAMS Support Regions

Assign county centroids to approximate 0.75° support regions and count how many counties fall into each region.


In [ ]:
# CAMS coarse-resolution context analysis using county centroids
print('=' * 70)
print('CAMS COARSE-RESOLUTION DIAGNOSTIC')
print('=' * 70)

cams_gdf = gpd.read_file(COUNTY_SHP_PATH)
if 'GEOID' in cams_gdf.columns:
    cams_gdf['FIPS'] = cams_gdf['GEOID'].astype(str).str.zfill(5)
elif {'STATEFP', 'COUNTYFP'}.issubset(cams_gdf.columns):
    cams_gdf['FIPS'] = cams_gdf['STATEFP'].astype(str).str.zfill(2) + cams_gdf['COUNTYFP'].astype(str).str.zfill(3)
else:
    raise ValueError('Could not determine county FIPS column from shapefile.')

cams_gdf = cams_gdf[cams_gdf['FIPS'].isin(study_counties)].copy()
if cams_gdf.crs is not None:
    cams_gdf = cams_gdf.to_crs(4326)

centroids = cams_gdf.geometry.centroid
cams_gdf['centroid_lon'] = centroids.x
cams_gdf['centroid_lat'] = centroids.y
cams_gdf['grid_lon'] = np.floor(cams_gdf['centroid_lon'] / 0.75) * 0.75
cams_gdf['grid_lat'] = np.floor(cams_gdf['centroid_lat'] / 0.75) * 0.75
cams_gdf['grid_id'] = cams_gdf['grid_lat'].round(2).astype(str) + '_' + cams_gdf['grid_lon'].round(2).astype(str)

all_grid_counts = cams_gdf.groupby('grid_id').size().reset_index(name='County Count')
all_grid_counts['Scope'] = 'All Study Counties'

test_counties_fips = model_b_test_predictions['Fips'].astype(str).str.zfill(5).unique()
test_grid_counts = cams_gdf[cams_gdf['FIPS'].isin(test_counties_fips)].groupby('grid_id').size().reset_index(name='County Count')
test_grid_counts['Scope'] = 'Model B Test Counties'

grid_counts_combined = pd.concat([all_grid_counts, test_grid_counts], ignore_index=True)
grid_counts_combined.to_csv(OUTPUT_DIR / 'cams_grid_cell_counts.csv', index=False)
record_artifact(OUTPUT_DIR / 'cams_grid_cell_counts.csv', 'Section 9', 'County counts per approximate CAMS support region', False)


### 9.2 Summary Tables and Histogram

Summarize the county-per-region distribution for all study counties and for the Model B test counties, then plot the overall histogram.


In [ ]:
grid_summary_df = pd.DataFrame([
    {
        'Scope': 'All Study Counties',
        'Num Support Regions': int(len(all_grid_counts)),
        'Mean Counties per Region': float(all_grid_counts['County Count'].mean()),
        'Median Counties per Region': float(all_grid_counts['County Count'].median()),
        'Max Counties per Region': int(all_grid_counts['County Count'].max()),
    },
    {
        'Scope': 'Model B Test Counties',
        'Num Support Regions': int(len(test_grid_counts)),
        'Mean Counties per Region': float(test_grid_counts['County Count'].mean()),
        'Median Counties per Region': float(test_grid_counts['County Count'].median()),
        'Max Counties per Region': int(test_grid_counts['County Count'].max()),
    }
])
grid_summary_df.to_csv(OUTPUT_DIR / 'cams_grid_summary.csv', index=False)
record_artifact(OUTPUT_DIR / 'cams_grid_summary.csv', 'Section 9', 'Summary of approximate CAMS support regions', True)

fig, ax = plt.subplots(figsize=(DOUBLE_COL_WIDTH, DOUBLE_COL_WIDTH * 0.6))
sns.histplot(all_grid_counts['County Count'], bins=30, color='slateblue', ax=ax)
ax.set_xlabel('Counties per Approximate 0.75° Support Region', fontsize=FONT_SIZE)
ax.set_ylabel('Frequency', fontsize=FONT_SIZE)
ax.set_title('Approximate County Counts per CAMS Support Region', fontsize=FONT_SIZE, fontweight='bold')
ax.grid(axis='y', linewidth=0.15, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'cams_grid_histogram.png', dpi=DPI, bbox_inches='tight')
plt.show()
record_artifact(OUTPUT_DIR / 'cams_grid_histogram.png', 'Section 9', 'Histogram of counties per approximate CAMS support region', False)

display(grid_summary_df)


---
## Section 10: Livestock Robustness via No-Livestock Sensitivity

This section directly tests whether the livestock interpolation issue materially changes the atmospheric result by refitting the revised model without livestock features.


### 10.1 Refit the Revised Model Without Livestock Variables

Drop the livestock features, refit the model using the canonical Model B hyperparameters, and compute headline performance metrics.


In [ ]:
# No-livestock sensitivity using fixed Model B best hyperparameters
print('=' * 70)
print('NO-LIVESTOCK SENSITIVITY ANALYSIS')
print('=' * 70)

X_train_no_livestock = X_train_model_b.drop(columns=LIVESTOCK_FEATURES)
X_test_no_livestock = X_test_model_b.drop(columns=LIVESTOCK_FEATURES)

no_livestock_model = build_fixed_param_model(best_params_model_b)
no_livestock_model.fit(X_train_no_livestock, y_train_model_b)

train_predictions_no_livestock = no_livestock_model.predict(X_train_no_livestock)
test_predictions_no_livestock = no_livestock_model.predict(X_test_no_livestock)
metrics_no_livestock = evaluate_predictions(
    y_train_model_b,
    train_predictions_no_livestock,
    y_test_model_b,
    test_predictions_no_livestock,
    n_features=X_train_no_livestock.shape[1]
)


### 10.2 No-Livestock SHAP Ranking

Compute the SHAP ranking for the no-livestock sensitivity model so formaldehyde rank changes can be compared directly.


In [ ]:
_, shap_ranking_no_livestock = compute_shap_outputs(
    no_livestock_model,
    X_test_no_livestock,
    ranking_csv_path=OUTPUT_DIR / 'livestock_no_livestock_shap_ranking.csv',
    values_path=OUTPUT_DIR / 'livestock_no_livestock_shap_values.npy',
    summary_fig_path=None,
    bar_fig_path=None,
    max_display=20
)
record_artifact(OUTPUT_DIR / 'livestock_no_livestock_shap_ranking.csv', 'Section 10', 'No-livestock SHAP ranking table', False)
record_artifact(OUTPUT_DIR / 'livestock_no_livestock_shap_values.npy', 'Section 10', 'No-livestock SHAP values', False)

formaldehyde_rank_no_livestock = int(
    shap_ranking_no_livestock.loc[shap_ranking_no_livestock['Feature'] == FORMALDEHYDE_FEATURE, 'Rank'].iloc[0]
)


### 10.3 Comparison Tables

Save the side-by-side performance comparison and the top-10 feature comparison for the main and no-livestock models.


In [ ]:
livestock_metrics_comparison = pd.DataFrame([
    {
        'Model': 'Model B (all features)',
        'Num Features': X_train_model_b.shape[1],
        'Test R²': metrics_model_b['test_r2'],
        'Test RMSE': metrics_model_b['test_rmse'],
        'Test MAE': metrics_model_b['test_mae'],
        'Formaldehyde Rank': model_b_formaldehyde_rank,
    },
    {
        'Model': 'No-livestock sensitivity',
        'Num Features': X_train_no_livestock.shape[1],
        'Test R²': metrics_no_livestock['test_r2'],
        'Test RMSE': metrics_no_livestock['test_rmse'],
        'Test MAE': metrics_no_livestock['test_mae'],
        'Formaldehyde Rank': formaldehyde_rank_no_livestock,
    }
])
livestock_metrics_comparison.to_csv(OUTPUT_DIR / 'livestock_sensitivity_metrics_comparison.csv', index=False)
record_artifact(OUTPUT_DIR / 'livestock_sensitivity_metrics_comparison.csv', 'Section 10', 'Model B vs no-livestock metrics comparison', True)

livestock_top10_comparison = pd.DataFrame({
    'Rank': np.arange(1, 11),
    'Model B Top 10': clean_display_labels(shap_ranking_model_b['Feature'].head(10).tolist()),
    'No-Livestock Top 10': clean_display_labels(shap_ranking_no_livestock['Feature'].head(10).tolist())
})
livestock_top10_comparison.to_csv(OUTPUT_DIR / 'livestock_top10_feature_comparison.csv', index=False)
record_artifact(OUTPUT_DIR / 'livestock_top10_feature_comparison.csv', 'Section 10', 'Top 10 feature comparison with and without livestock', False)

display(livestock_metrics_comparison)
display(livestock_top10_comparison)


---
## Section 11: Updated Ablation Study from Model B Rankings

This section recomputes the top 20 / top 10 / top 5 models using the Model B SHAP ranking as the feature selection basis.


### 11.1 Define Reduced Feature Sets

Construct the top-20, top-10, and top-5 feature sets from the filtered Model B SHAP ranking.


In [ ]:
# Updated ablation study using Model B SHAP ranking
print('=' * 70)
print('UPDATED ABLATION STUDY FROM MODEL B RANKINGS')
print('=' * 70)

ablation_feature_sets = {
    'Top 20': shap_ranking_model_b['Feature'].head(20).tolist(),
    'Top 10': shap_ranking_model_b['Feature'].head(10).tolist(),
    'Top 5': shap_ranking_model_b['Feature'].head(5).tolist(),
}

ablation_results = [
    {
        'Feature Set': 'All Features',
        'Num Features': int(X_train_model_b.shape[1]),
        'Train R²': metrics_model_b['train_r2'],
        'Test R²': metrics_model_b['test_r2'],
        'Train RMSE': metrics_model_b['train_rmse'],
        'Test RMSE': metrics_model_b['test_rmse'],
        'Train MAE': metrics_model_b['train_mae'],
        'Test MAE': metrics_model_b['test_mae'],
    }
]


### 11.2 Fit Reduced Models

Refit the reduced-feature ablation models with their own Bayesian optimization runs and collect the resulting performance metrics.


In [ ]:
for feature_set_name, feature_list in ablation_feature_sets.items():
    print('\n' + '=' * 70)
    print(f'ABLATION MODEL: {feature_set_name}')
    print('=' * 70)
    print('Selected features:')
    for idx, feature in enumerate(clean_display_labels(feature_list), start=1):
        print(f'  {idx:2}. {feature}')

    X_train_subset = X_train_model_b[feature_list].copy()
    X_test_subset = X_test_model_b[feature_list].copy()

    bayes_subset, best_subset_model = run_bayes_xgb(
        X_train_subset,
        y_train_model_b,
        groups_train_model_b,
        section_name=f'{feature_set_name} ablation model'
    )

    subset_train_predictions = best_subset_model.predict(X_train_subset)
    subset_test_predictions = best_subset_model.predict(X_test_subset)
    subset_metrics = evaluate_predictions(
        y_train_model_b,
        subset_train_predictions,
        y_test_model_b,
        subset_test_predictions,
        n_features=X_train_subset.shape[1]
    )

    feature_slug = feature_set_name.lower().replace(' ', '_')
    save_json(_json_safe(bayes_subset.best_params_), OUTPUT_DIR / f'ablation_{feature_slug}_best_params.json')
    record_artifact(OUTPUT_DIR / f'ablation_{feature_slug}_best_params.json', 'Section 11', f'Best parameters for {feature_set_name}', False)

    subset_metrics_df = save_metrics_table(
        subset_metrics,
        OUTPUT_DIR / f'ablation_{feature_slug}_metrics.csv',
        title=f'{feature_set_name} Metrics Table'
    )
    record_artifact(OUTPUT_DIR / f'ablation_{feature_slug}_metrics.csv', 'Section 11', f'{feature_set_name} ablation metrics', False)

    ablation_results.append({
        'Feature Set': feature_set_name,
        'Num Features': int(X_train_subset.shape[1]),
        'Train R²': subset_metrics['train_r2'],
        'Test R²': subset_metrics['test_r2'],
        'Train RMSE': subset_metrics['train_rmse'],
        'Test RMSE': subset_metrics['test_rmse'],
        'Train MAE': subset_metrics['train_mae'],
        'Test MAE': subset_metrics['test_mae'],
    })


### 11.3 Ablation Comparison Table

Assemble the all-features and reduced-model metrics into a single comparison table for the manuscript.


In [ ]:
ablation_df = pd.DataFrame(ablation_results)
ablation_df.to_csv(OUTPUT_DIR / 'ablation_model_comparison.csv', index=False)
record_artifact(OUTPUT_DIR / 'ablation_model_comparison.csv', 'Section 11', 'Updated ablation comparison table', True)


### 11.4 Ablation Performance Curve

Plot how test performance changes as the feature set shrinks from the full revised model to the top-20, top-10, and top-5 subsets.


In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 6))
color1 = '#2563eb'
ax1.set_xlabel('Number of Features', fontsize=FONT_SIZE)
ax1.set_ylabel('$R^2$ Score', color=color1, fontsize=FONT_SIZE)
line1 = ax1.plot(ablation_df['Num Features'], ablation_df['Test R²'], marker='o', markersize=10,
                 linewidth=2.5, color=color1, label='Test $R^2$')
ax1.tick_params(axis='y', labelcolor=color1)
r2_min = ablation_df['Test R²'].min()
r2_max = ablation_df['Test R²'].max()
r2_padding = (r2_max - r2_min) * 0.15 if r2_max > r2_min else 0.05
ax1.set_ylim([r2_min - r2_padding, min(1.0, r2_max + r2_padding)])
ax1.grid(axis='both', linewidth=0.3, alpha=0.4)

ax2 = ax1.twinx()
color2 = '#dc2626'
ax2.set_ylabel('RMSE (years)', color=color2, fontsize=FONT_SIZE)
line2 = ax2.plot(ablation_df['Num Features'], ablation_df['Test RMSE'], marker='s', markersize=10,
                 linewidth=2.5, color=color2, label='Test RMSE')
ax2.tick_params(axis='y', labelcolor=color2)
rmse_min = ablation_df['Test RMSE'].min()
rmse_max = ablation_df['Test RMSE'].max()
rmse_padding = (rmse_max - rmse_min) * 0.15 if rmse_max > rmse_min else 0.05
ax2.set_ylim([max(0, rmse_min - rmse_padding), rmse_max + rmse_padding])

ax1.set_title('Ablation Study: Model Performance vs. Number of Features', fontsize=FONT_SIZE + 1, fontweight='bold', pad=15)
lines = line1 + line2
labels = [line.get_label() for line in lines]
ax1.legend(lines, labels, loc='upper right', fontsize=FONT_SIZE - 1, framealpha=0.9)
ax1.set_xticks(ablation_df['Num Features'])
ax1.set_xticklabels(ablation_df['Num Features'].astype(int))
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'ablation_performance_curve.png', dpi=DPI, bbox_inches='tight')
plt.show()
record_artifact(OUTPUT_DIR / 'ablation_performance_curve.png', 'Section 11', 'Updated ablation figure', True)

display(ablation_df)


---
## Section 12: Artifact Manifest and Reviewer Mapping

This final section creates a compact handoff table showing what was saved, what belongs in the paper, and which reviewer concerns each output addresses.


### 12.1 Final Artifact Manifest and Reviewer Mapping

Summarize every saved artifact, flag which outputs are paper-facing, and map the major reviewer comments to the notebook outputs that address them.


In [ ]:
# Final artifact manifest and reviewer mapping
record_artifact(OUTPUT_DIR / 'artifact_manifest.csv', 'Section 12', 'Notebook artifact manifest', False)
record_artifact(OUTPUT_DIR / 'reviewer_mapping.csv', 'Section 12', 'Reviewer-to-output mapping', False)

artifact_manifest_df = pd.DataFrame(artifact_manifest)
artifact_manifest_df = artifact_manifest_df.sort_values(by=['section', 'path']).reset_index(drop=True)
artifact_manifest_df.to_csv(OUTPUT_DIR / 'artifact_manifest.csv', index=False)

reviewer_mapping_df = pd.DataFrame([
    {'Reviewer Comment': 'R1-2 / R2-M2', 'Notebook Output': 'model_b_metrics.csv, model_b_shap_ranking.csv', 'Paper Facing': 'Yes'},
    {'Reviewer Comment': 'R2-M1', 'Notebook Output': 'confounding_formaldehyde_poverty_quartiles.png, confounding_partial_correlation.csv', 'Paper Facing': 'Yes / Response'},
    {'Reviewer Comment': 'R2-M5', 'Notebook Output': 'temporal_metrics.csv, ablation_model_comparison.csv', 'Paper Facing': 'Yes'},
    {'Reviewer Comment': 'R2-M6', 'Notebook Output': 'spatial_morans_i.csv', 'Paper Facing': 'Yes'},
    {'Reviewer Comment': 'R2-m4', 'Notebook Output': 'stability_shap_ranks.csv, stability_shap_summary.json', 'Paper Facing': 'Response'},
    {'Reviewer Comment': 'R2-m5', 'Notebook Output': 'convergence_model_b.csv, convergence_model_b.png', 'Paper Facing': 'Response'},
    {'Reviewer Comment': 'R2-M3a', 'Notebook Output': 'cams_grid_summary.csv, cams_grid_cell_counts.csv', 'Paper Facing': 'Yes'},
    {'Reviewer Comment': 'R1-3 / R2-M7', 'Notebook Output': 'livestock_sensitivity_metrics_comparison.csv, livestock_top10_feature_comparison.csv', 'Paper Facing': 'Yes / Response'},
])
reviewer_mapping_df.to_csv(OUTPUT_DIR / 'reviewer_mapping.csv', index=False)

print('=' * 70)
print('NOTEBOOK 12 COMPLETE: ARTIFACT MANIFEST')
print('=' * 70)
print(f'Total artifacts recorded: {len(artifact_manifest_df)}')
print('\nPaper-facing artifacts:')
display(artifact_manifest_df[artifact_manifest_df['paper_facing']].reset_index(drop=True))
print('\nReviewer mapping:')
display(reviewer_mapping_df)
print('\nExplicitly excluded from notebook 12:')
print('  - OMI/TROPOMI direct comparison')
print('  - FoT recomputation from raw atmospheric data')
